# Milestone 2: Data Quality dan Profiling

**Dataset:** TMDB Movies Dataset (`TMDB_movie_dataset_v11.csv`)  
**Tujuan:** melanjutkan Milestone 1 dengan profiling kualitas data, penanganan missing values, pemeriksaan outlier, deduplikasi berdasarkan ID film, validasi, dan penyimpanan data bersih.

Notebook ini menggunakan **Polars** untuk pengolahan data dan **DuckDB `SUMMARIZE`** untuk profiling. Data mentah di `data/raw/` tidak diubah; keluaran disimpan terpisah di `data/processed/`.


In [1]:
from pathlib import Path
import polars as pl
import duckdb

# Mendukung eksekusi dari root repository atau folder notebooks/
CWD = Path.cwd().resolve()
if (CWD / "data" / "raw").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "data" / "raw").exists():
    PROJECT_ROOT = CWD.parent
else:
    PROJECT_ROOT = CWD

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "TMDB_movie_dataset_v11.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "TMDB_movie_dataset_clean.parquet"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH}. Unduh TMDB_movie_dataset_v11.csv "
        "dan letakkan di folder data/raw/."
    )

# Membaca CSV; nilai kosong umum dikenali sebagai null.
df = pl.read_csv(
    DATA_PATH,
    infer_schema_length=10000,
    null_values=["", "NA", "N/A", "null", "None"],
    try_parse_dates=False,
    ignore_errors=True,
    low_memory=True,
)
print(f"Ukuran awal: {df.height:,} baris x {df.width} kolom")
print("Kolom:", df.columns)
df.head()


Ukuran awal: 1,164,598 baris x 24 kolom
Kolom: ['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date', 'revenue', 'runtime', 'adult', 'backdrop_path', 'budget', 'homepage', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'tagline', 'genres', 'production_companies', 'production_countries', 'spoken_languages', 'keywords']


id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,budget,homepage,imdb_id,original_language,original_title,overview,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages,keywords
i64,str,f64,i64,str,str,i64,i64,bool,str,i64,str,str,str,str,str,f64,str,str,str,str,str,str,str
27205,"""Inception""",8.364,34495,"""Released""","""2010-07-15""",825532764,148,false,"""/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.j…",160000000,"""https://www.warnerbros.com/mov…","""tt1375666""","""en""","""Inception""","""Cobb, a skilled thief who comm…",83.952,"""/oYuLEt3zVCKq57qu2F8dT7NIa6f.j…","""Your mind is the scene of the …","""Action, Science Fiction, Adven…","""Legendary Pictures, Syncopy, W…","""United Kingdom, United States …","""English, French, Japanese, Swa…","""rescue, mission, dream, airpla…"
157336,"""Interstellar""",8.417,32571,"""Released""","""2014-11-05""",701729206,169,false,"""/pbrkL804c8yAv3zBZR4QPEafpAR.j…",165000000,"""http://www.interstellarmovie.n…","""tt0816692""","""en""","""Interstellar""","""The adventures of a group of e…",140.241,"""/gEU2QniE6E77NI6lCU6MxlNBvIx.j…","""Mankind was born on Earth. It …","""Adventure, Drama, Science Fict…","""Legendary Pictures, Syncopy, L…","""United Kingdom, United States …","""English""","""rescue, future, spacecraft, ra…"
155,"""The Dark Knight""",8.512,30619,"""Released""","""2008-07-16""",1004558444,152,false,"""/nMKdUUepR0i5zn0y1T4CsSB5chy.j…",185000000,"""https://www.warnerbros.com/mov…","""tt0468569""","""en""","""The Dark Knight""","""Batman raises the stakes in hi…",130.643,"""/qJ2tW6WMUDux911r6m7haRef0WH.j…","""Welcome to a world without rul…","""Drama, Action, Crime, Thriller""","""DC Comics, Legendary Pictures,…","""United Kingdom, United States …","""English, Mandarin""","""joker, sadism, chaos, secret i…"
19995,"""Avatar""",7.573,29815,"""Released""","""2009-12-15""",2923706026,162,false,"""/vL5LR6WdxWPjLPFRLe133jXWsh5.j…",237000000,"""https://www.avatar.com/movies/…","""tt0499549""","""en""","""Avatar""","""In the 22nd century, a paraple…",79.932,"""/kyeqWdyUXW608qlYkRqosgbbJyK.j…","""Enter the world of Pandora.""","""Action, Adventure, Fantasy, Sc…","""Dune Entertainment, Lightstorm…","""United States of America, Unit…","""English, Spanish""","""future, society, culture clash…"
24428,"""The Avengers""",7.71,29166,"""Released""","""2012-04-25""",1518815515,143,false,"""/9BBTo63ANSmhC4e6r62OJFuK2GL.j…",220000000,"""https://www.marvel.com/movies/…","""tt0848228""","""en""","""The Avengers""","""When an unexpected enemy emerg…",98.082,"""/RYMX2wcKCBAr24UyPD7xwmjaTn.jp…","""Some assembly required.""","""Science Fiction, Action, Adven…","""Marvel Studios""","""United States of America""","""English, Hindi, Russian""","""new york city, superhero, shie…"


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [2]:
connection = duckdb.connect()
connection.register("dataset", df.to_arrow())

# SUMMARIZE merangkum tipe data, statistik, null, dan distinct untuk setiap kolom.
profil = connection.execute("SUMMARIZE SELECT * FROM dataset").pl()
profil


column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""id""","""BIGINT""","""2""","""1420551""",1122523,"""756600.6537199961""","""404105.3784664535""","""417670""","""758870""","""1116662""",1164598,0.00
"""title""","""VARCHAR""",""" ""","""😤 insect james 😤""",811835,null,null,null,null,null,1164598,0.00
"""vote_average""","""DOUBLE""","""0.0""","""10.0""",5562,"""1.840550935172481""","""3.0037313404900736""","""0.0""","""0.0""","""4.543681558741255""",1164598,0.00
"""vote_count""","""BIGINT""","""0""","""34495""",3672,"""18.418174339986845""","""314.677114223334""","""0""","""0""","""1""",1164598,0.00
"""status""","""VARCHAR""","""Canceled""","""Rumored""",5,null,null,null,null,null,1164598,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""genres""","""VARCHAR""","""Action""","""Western, War, Thriller""",13953,null,null,null,null,null,1164598,40.52
"""production_companies""","""VARCHAR""","""""A ME AND YOU MOTION PICTURE""""","""ＶＩＬＩＳ ＩＮＣ, Shintoho Pictures""",251311,null,null,null,null,null,1164598,55.10
"""production_countries""","""VARCHAR""","""Afghanistan""","""Zimbabwe, Spain, South Africa,…",8165,null,null,null,null,null,1164598,44.78


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [3]:
jumlah_baris = df.height
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name="kolom", column_names=["jumlah_null"])
    .with_columns((pl.col("jumlah_null") / max(jumlah_baris, 1) * 100).round(2).alias("persen_null"))
    .filter(pl.col("jumlah_null") > 0)
    .sort("jumlah_null", descending=True)
)
null_summary


kolom,jumlah_null,persen_null
str,u32,f64
"""homepage""",1041847,89.46
"""tagline""",1001414,85.99
"""backdrop_path""",857714,73.65
"""keywords""",852365,73.19
"""production_companies""",641725,55.1
…,…,…
"""poster_path""",373033,32.03
"""overview""",240202,20.63
"""release_date""",199083,17.09


## Strategi yang digunakan

- `title` kosong: baris dihapus karena judul adalah identitas utama yang dibutuhkan untuk mengenali film.
- `id` kosong: baris dihapus karena ID menjadi kunci deduplikasi.
- Kolom teks deskriptif (`overview`, `tagline`, `genres`, `keywords`, dan kolom teks terkait) yang kosong **tidak** otomatis diisi dengan modus; nilai kosong diganti string kosong hanya untuk kolom teks tersebut agar tetap jelas bahwa informasi tidak tersedia.
- Nilai numerik seperti `budget` dan `revenue` tidak diisi 0 atau median, karena nilai kosong tidak selalu berarti benar-benar nol dan imputasi dapat mengubah makna data.
- Kolom numerik lainnya dibiarkan null kecuali aturan domain di atas secara eksplisit menyebutkannya.


In [4]:
# Bersihkan spasi pada kolom teks dan hapus baris tanpa identitas film.
kolom_teks = [c for c, dtype in zip(df.columns, df.dtypes) if dtype == pl.String]
df_clean = df.with_columns([
    pl.col(c).str.strip_chars().alias(c) for c in kolom_teks
])

# String kosong dianggap tidak memiliki nilai untuk kolom identitas.
for c in ["id", "title"]:
    if c in df_clean.columns and df_clean.schema[c] == pl.String:
        df_clean = df_clean.with_columns(
            pl.when(pl.col(c).str.strip_chars() == "").then(None).otherwise(pl.col(c)).alias(c)
        )

if "id" in df_clean.columns:
    df_clean = df_clean.filter(pl.col("id").is_not_null())
if "title" in df_clean.columns:
    df_clean = df_clean.filter(pl.col("title").is_not_null())

# Untuk kolom teks deskriptif, null diganti string kosong; tidak mengarang nilai kategorik.
kolom_deskriptif = [c for c in ["overview", "tagline", "genres", "keywords", "homepage",
                                  "imdb_id", "backdrop_path", "poster_path",
                                  "production_companies", "production_countries", "spoken_languages"]
                    if c in df_clean.columns and df_clean.schema[c] == pl.String]
if kolom_deskriptif:
    df_clean = df_clean.with_columns([pl.col(c).fill_null("") for c in kolom_deskriptif])

print(f"Total null sebelum: {df.null_count().sum_horizontal().sum():,}")
print(f"Total null sesudah strategi: {df_clean.null_count().sum_horizontal().sum():,}")
print(f"Baris dihapus karena id/title kosong: {df.height - df_clean.height:,}")


Total null sebelum: 7,257,828
Total null sesudah strategi: 199,072
Baris dihapus karena id/title kosong: 16


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [5]:
# Gunakan popularity sebagai contoh numerik untuk deteksi outlier IQR.
# Outlier popularity hanya di-winsorize (clip), bukan menghapus film.
KOLOM_NUMERIK = "popularity"
if KOLOM_NUMERIK not in df_clean.columns or not df_clean.schema[KOLOM_NUMERIK].is_numeric():
    raise ValueError(f"Kolom numerik {KOLOM_NUMERIK!r} tidak ditemukan/ bukan numerik. Periksa skema dataset.")

q1 = df_clean[KOLOM_NUMERIK].quantile(0.25)
q3 = df_clean[KOLOM_NUMERIK].quantile(0.75)
iqr = q3 - q1
batas_bawah = q1 - 1.5 * iqr
batas_atas = q3 + 1.5 * iqr

jumlah_outlier = df_clean.filter(
    pl.col(KOLOM_NUMERIK).is_not_null() &
    ((pl.col(KOLOM_NUMERIK) < batas_bawah) | (pl.col(KOLOM_NUMERIK) > batas_atas))
).height
print(f"Q1={q1}, Q3={q3}, IQR={iqr}")
print(f"Batas bawah: {batas_bawah}; batas atas: {batas_atas}")
print(f"Jumlah nilai popularity di luar batas IQR: {jumlah_outlier:,}")

df_outlier_handled = df_clean.with_columns(
    pl.col(KOLOM_NUMERIK).clip(lower_bound=batas_bawah, upper_bound=batas_atas).alias(KOLOM_NUMERIK)
)


Q1=0.6, Q3=0.874, IQR=0.274
Batas bawah: 0.18899999999999995; batas atas: 1.2850000000000001
Jumlah nilai popularity di luar batas IQR: 334,703


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [6]:
# ID TMDB merupakan kunci logis film. Simpan kemunculan pertama untuk ID yang berulang.
KOLOM_KUNCI = "id"
if KOLOM_KUNCI not in df_outlier_handled.columns:
    raise ValueError("Kolom 'id' tidak ditemukan; deduplikasi tidak dapat dilakukan dengan aman.")

total_baris = df_outlier_handled.height
jumlah_id_unik = df_outlier_handled[KOLOM_KUNCI].n_unique()
print(f"Total baris sebelum deduplikasi: {total_baris:,}")
print(f"ID unik: {jumlah_id_unik:,}")
print(f"Perkiraan baris dengan ID berulang: {total_baris - jumlah_id_unik:,}")

df_dedup = df_outlier_handled.unique(
    subset=[KOLOM_KUNCI], keep="first", maintain_order=True
)
print(f"Total baris sesudah deduplikasi: {df_dedup.height:,}")


Total baris sebelum deduplikasi: 1,164,582
ID unik: 1,163,770
Perkiraan baris dengan ID berulang: 812
Total baris sesudah deduplikasi: 1,163,770


## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [7]:
# Validasi kualitas data setelah cleaning.
connection.register("dataset_bersih", df_dedup.to_arrow())
validasi = connection.execute("""
    SELECT
        COUNT(*) AS total_baris,
        COUNT(*) FILTER (WHERE id IS NULL) AS id_null,
        COUNT(*) FILTER (WHERE title IS NULL OR trim(title) = '') AS title_kosong,
        COUNT(DISTINCT id) AS jumlah_id_unik,
        COUNT(*) FILTER (WHERE popularity < ? OR popularity > ?) AS popularity_di_luar_batas
    FROM dataset_bersih
""", [batas_bawah, batas_atas]).pl()
validasi


total_baris,id_null,title_kosong,jumlah_id_unik,popularity_di_luar_batas
i64,i64,i64,i64,i64
1163770,0,0,1163770,0


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [8]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH, compression="zstd")

print(f"Baris sebelum cleaning: {df.height:,}")
print(f"Baris sesudah cleaning: {df_dedup.height:,}")
print(f"Perubahan jumlah baris: {df.height - df_dedup.height:,}")
print(f"Data bersih tersimpan di: {OUTPUT_PATH}")
print("Dataset mentah tetap tidak diubah.")


Baris sebelum cleaning: 1,164,598
Baris sesudah cleaning: 1,163,770
Perubahan jumlah baris: 828
Data bersih tersimpan di: /home/jovyan/work/data/processed/TMDB_movie_dataset_clean.parquet
Dataset mentah tetap tidak diubah.


## Catatan Milestone 2

Setelah seluruh cell dijalankan, isi kesimpulan berdasarkan output aktual:

1. **Dimensi kualitas paling bermasalah:** lihat tabel missing values pada bagian 2 dan hasil `SUMMARIZE` pada bagian 1.
2. **Missing values:** baris tanpa `id` atau `title` dihapus; kolom teks deskriptif yang kosong diisi string kosong; nilai finansial tidak diimputasi agar nilai yang tidak diketahui tidak dianggap nol.
3. **Outlier:** `popularity` diperiksa menggunakan IQR dan nilai di luar batas di-clip. Metode ini mengurangi pengaruh nilai ekstrem tanpa menghapus baris film.
4. **Duplikat:** ID film `id` digunakan sebagai kunci dan hanya satu baris dipertahankan per ID.
5. **Validasi:** periksa bahwa `id_null`, `title_kosong`, dan `popularity_di_luar_batas` bernilai 0, serta `jumlah_id_unik` sama dengan `total_baris`.
6. **Output:** data bersih disimpan sebagai Parquet di `data/processed/TMDB_movie_dataset_clean.parquet` untuk digunakan pada Milestone 3.

> Catatan: angka hasil cleaning harus diambil dari output notebook setelah dijalankan pada file dataset lokal, bukan diperkirakan sebelumnya.
